# Identifying Fraudulent Activities — XGBoost Workflow (2026)

This notebook rebuilds the e-commerce fraud project around **XGBoost** while preserving the leakage-safe, time-aware, calibrated, and cost-sensitive design of the modernized workflow.

**Business task:** At a user's first purchase, estimate the probability that the transaction is fraudulent so the company can approve it, request stronger authentication, send it to review, delay fulfillment, or block it under a documented policy.

## Why XGBoost is a good candidate

Fraud risk often depends on nonlinear combinations such as:

- a very short signup-to-purchase delay **and** a reused device;
- an unusual transaction value **and** a risky historical IP pattern;
- different timing behavior across acquisition sources or countries;
- threshold effects that a linear model cannot represent naturally.

XGBoost is effective for this type of tabular data because it builds regularized decision-tree ensembles, handles missing numeric values, supports imbalanced-class weighting, and can learn feature interactions without manually specifying them.

## Main upgrades in this version

- Uses XGBoost's histogram tree method and native categorical splits.
- Requires XGBoost 3.1 or newer so pandas category encodings can be remembered and recoded consistently.
- Maps rare and unseen categories to an explicit `__OTHER__` level.
- Computes `scale_pos_weight` separately inside every chronological training fold.
- Uses early stopping only inside rolling training-period validation.
- Chooses the final boosting-round count from the historical validation folds, then refits on all training data.
- Calibrates raw XGBoost probabilities on a later, disjoint period.
- Tunes cost and review-capacity thresholds on another later period.
- Evaluates once on an untouched future test period.
- Adds permutation importance, XGBoost gain importance, and native SHAP contribution summaries.
- Saves the native categorical XGBoost model in JSON format plus a separate policy/calibration artifact.

> A fraud score is decision support, not proof that a customer committed fraud. High-risk cases should follow a documented review, explanation, and appeal process.

## 1. Environment

The notebook uses XGBoost's scikit-learn interface and native categorical support. XGBoost 3.1 introduced automatic category recoding for pandas DataFrames, which helps keep category encodings consistent at prediction time.

The default device is CPU for portability. Set the environment variable `XGB_DEVICE=cuda` before execution to use a compatible GPU environment.

Uncomment the next line only when packages are missing.

In [ ]:
# %pip install -U pandas numpy scipy scikit-learn xgboost matplotlib seaborn joblib nbformat

In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
import warnings
from pathlib import Path
from typing import Iterable

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy
import sklearn
import xgboost as xgb
from IPython.display import display
from packaging.version import Version

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.calibration import CalibrationDisplay
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(context='notebook', style='whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

RANDOM_STATE = 42
TARGET = 'class'
USE_SENSITIVE_FEATURES = False  # `sex` is retained for auditing, not training, by default.
XGB_DEVICE = os.getenv('XGB_DEVICE', 'cpu')

if Version(xgb.__version__) < Version('3.1.0'):
    raise RuntimeError(
        f'This notebook requires xgboost>=3.1.0 for native-category auto-recoding; found {xgb.__version__}.'
    )

print({
    'python': platform.python_version(),
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'scipy': scipy.__version__,
    'scikit_learn': sklearn.__version__,
    'xgboost': xgb.__version__,
    'xgb_device': XGB_DEVICE,
})

## 2. Load the two source files

Expected files:

- `Fraud.csv`
- `IpAddress_to_Country.csv`

The loader checks environment variables first, then common local folders. This removes the hard-coded local path used by many older notebooks.

Optional environment variables:

```bash
export FRAUD_DATA_PATH=/path/to/Fraud.csv
export IP_COUNTRY_PATH=/path/to/IpAddress_to_Country.csv
```

In [ ]:
def find_data_file(filename: str, env_var: str) -> Path:
    candidates: list[Path] = []
    if os.getenv(env_var):
        candidates.append(Path(os.environ[env_var]).expanduser())

    cwd = Path.cwd()
    candidates.extend([
        cwd / filename,
        cwd / 'data' / filename,
        cwd.parent / 'data' / filename,
        Path('/mnt/data') / filename,
    ])

    for path in candidates:
        if path.exists() and path.is_file():
            return path.resolve()

    checked = '\n'.join(f'  - {p}' for p in candidates)
    raise FileNotFoundError(
        f'Could not locate {filename}. Checked:\n{checked}\n'
        f'Place the file beside this notebook, in data/, or set {env_var}.'
    )

fraud_path = find_data_file('Fraud.csv', 'FRAUD_DATA_PATH')
ip_country_path = find_data_file('IpAddress_to_Country.csv', 'IP_COUNTRY_PATH')

fraud_raw = pd.read_csv(fraud_path)
ip_country_raw = pd.read_csv(ip_country_path)

print(f'Fraud data: {fraud_path} — {fraud_raw.shape[0]:,} rows')
print(f'IP ranges:  {ip_country_path} — {ip_country_raw.shape[0]:,} rows')
display(fraud_raw.head())
display(ip_country_raw.head())

## 3. Validate the schema before analysis

Failing early is safer than silently training on renamed, missing, or incorrectly typed columns.

In [ ]:
FRAUD_REQUIRED = {
    'user_id', 'signup_time', 'purchase_time', 'purchase_value', 'device_id',
    'source', 'browser', 'sex', 'age', 'ip_address', TARGET,
}
IP_REQUIRED = {'lower_bound_ip_address', 'upper_bound_ip_address', 'country'}

missing_fraud = FRAUD_REQUIRED.difference(fraud_raw.columns)
missing_ip = IP_REQUIRED.difference(ip_country_raw.columns)
if missing_fraud or missing_ip:
    raise ValueError({
        'missing_fraud_columns': sorted(missing_fraud),
        'missing_ip_range_columns': sorted(missing_ip),
    })

schema_summary = pd.DataFrame({
    'dtype': fraud_raw.dtypes.astype(str),
    'missing_n': fraud_raw.isna().sum(),
    'missing_pct': fraud_raw.isna().mean(),
    'unique_n': fraud_raw.nunique(dropna=False),
}).sort_values('missing_pct', ascending=False)
display(schema_summary)

## 4. Clean records conservatively

Rules used here:

- Dates must parse successfully.
- The target must be 0 or 1.
- Purchase time cannot be before signup time.
- Purchase value cannot be negative.
- Age must be between 0 and 100.
- Exact duplicate rows are removed.

Unusual records are counted before removal so the data-quality impact is visible.

In [ ]:
def clean_fraud_data(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = raw.copy()
    df['signup_time'] = pd.to_datetime(df['signup_time'], errors='coerce', utc=False)
    df['purchase_time'] = pd.to_datetime(df['purchase_time'], errors='coerce', utc=False)

    for col in ['purchase_value', 'age', 'ip_address', TARGET]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    flags = pd.DataFrame(index=df.index)
    flags['bad_signup_time'] = df['signup_time'].isna()
    flags['bad_purchase_time'] = df['purchase_time'].isna()
    flags['bad_target'] = ~df[TARGET].isin([0, 1])
    flags['purchase_before_signup'] = df['purchase_time'] < df['signup_time']
    flags['negative_purchase_value'] = df['purchase_value'] < 0
    flags['invalid_age'] = ~df['age'].between(0, 100, inclusive='both')
    flags['exact_duplicate'] = df.duplicated(keep='first')

    report = pd.DataFrame({
        'issue': flags.columns,
        'rows': [int(flags[c].sum()) for c in flags.columns],
    })
    report['pct_of_raw'] = report['rows'] / max(len(df), 1)

    invalid = flags.any(axis=1)
    clean = df.loc[~invalid].copy()
    clean[TARGET] = clean[TARGET].astype('int8')
    clean['age'] = clean['age'].astype('float64')
    clean['purchase_value'] = clean['purchase_value'].astype('float64')
    clean['ip_address'] = clean['ip_address'].round().astype('Int64')

    # Standardize string categories while preserving unknown values.
    for col in ['device_id', 'source', 'browser', 'sex']:
        clean[col] = clean[col].astype('string').str.strip().fillna('Unknown')

    clean = clean.sort_values(['purchase_time', 'user_id']).reset_index(drop=True)
    return clean, report

fraud, quality_report = clean_fraud_data(fraud_raw)
display(quality_report)
print(f'Rows retained: {len(fraud):,} / {len(fraud_raw):,}')
print(f'Duplicate user IDs: {fraud["user_id"].duplicated().sum():,}')

## 5. Map IP addresses to countries efficiently

Each country row defines an inclusive numeric IP interval. For every transaction, `numpy.searchsorted` finds the last lower bound not exceeding the IP; the upper bound is then checked. Complexity is approximately `O(n log m)` rather than repeatedly scanning all ranges.

In [ ]:
def prepare_ip_ranges(raw: pd.DataFrame) -> pd.DataFrame:
    ranges = raw.copy()
    ranges['lower_bound_ip_address'] = pd.to_numeric(
        ranges['lower_bound_ip_address'], errors='coerce'
    )
    ranges['upper_bound_ip_address'] = pd.to_numeric(
        ranges['upper_bound_ip_address'], errors='coerce'
    )
    ranges['country'] = ranges['country'].astype('string').str.strip().fillna('Unknown')
    ranges = ranges.dropna(subset=['lower_bound_ip_address', 'upper_bound_ip_address'])
    ranges = ranges[
        ranges['lower_bound_ip_address'] <= ranges['upper_bound_ip_address']
    ].sort_values('lower_bound_ip_address').reset_index(drop=True)

    overlap = (
        ranges['lower_bound_ip_address'].iloc[1:].to_numpy()
        <= ranges['upper_bound_ip_address'].iloc[:-1].to_numpy()
    )
    if overlap.any():
        warnings.warn(f'{overlap.sum():,} adjacent IP ranges overlap; verify source data.')
    return ranges


def map_ip_to_country(ip: pd.Series, ranges: pd.DataFrame) -> pd.Series:
    lower = ranges['lower_bound_ip_address'].to_numpy(dtype='float64')
    upper = ranges['upper_bound_ip_address'].to_numpy(dtype='float64')
    countries = ranges['country'].astype(str).to_numpy()
    values = pd.to_numeric(ip, errors='coerce').to_numpy(dtype='float64')

    idx = np.searchsorted(lower, values, side='right') - 1
    valid_idx = idx >= 0
    safe_idx = np.clip(idx, 0, len(ranges) - 1)
    valid = valid_idx & np.isfinite(values) & (values <= upper[safe_idx])

    result = np.full(len(values), 'Unknown', dtype=object)
    result[valid] = countries[safe_idx[valid]]
    return pd.Series(result, index=ip.index, dtype='string')

ip_ranges = prepare_ip_ranges(ip_country_raw)
fraud['country'] = map_ip_to_country(fraud['ip_address'], ip_ranges)

print(f'Country match rate: {(fraud["country"] != "Unknown").mean():.2%}')
display(fraud[['ip_address', 'country']].head())

## 6. Build features available at decision time

A common leakage problem in the older approach is computing each device/IP's **final total count across the full dataset**. A transaction early in the year would then know about accounts that appear months later.

This version sorts transactions chronologically and uses `cumcount()`. Therefore:

- `device_prior_count` = earlier transactions using the same device.
- `ip_prior_count` = earlier transactions using the same IP.

For an online system, these values come from a feature store immediately before scoring. Raw `user_id`, `device_id`, and `ip_address` are never model inputs.

In [ ]:
def cyclic_encode(values: pd.Series, period: int, prefix: str) -> pd.DataFrame:
    angle = 2 * np.pi * values.astype(float) / period
    return pd.DataFrame({
        f'{prefix}_sin': np.sin(angle),
        f'{prefix}_cos': np.cos(angle),
    }, index=values.index)


def build_event_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(['purchase_time', 'user_id']).copy()

    # Known at the first purchase.
    gap_seconds = (out['purchase_time'] - out['signup_time']).dt.total_seconds()
    out['time_since_signup_hours'] = gap_seconds / 3600
    out['log_time_since_signup_hours'] = np.log1p(out['time_since_signup_hours'].clip(lower=0))
    out['purchase_value_log'] = np.log1p(out['purchase_value'].clip(lower=0))

    out['purchase_is_weekend'] = out['purchase_time'].dt.dayofweek.ge(5).astype('int8')
    out['signup_is_weekend'] = out['signup_time'].dt.dayofweek.ge(5).astype('int8')
    out['instant_purchase_10s'] = gap_seconds.le(10).astype('int8')
    out['instant_purchase_1h'] = gap_seconds.le(3600).astype('int8')

    for frame in [
        cyclic_encode(out['purchase_time'].dt.hour, 24, 'purchase_hour'),
        cyclic_encode(out['purchase_time'].dt.dayofweek, 7, 'purchase_dow'),
        cyclic_encode(out['signup_time'].dt.hour, 24, 'signup_hour'),
        cyclic_encode(out['signup_time'].dt.dayofweek, 7, 'signup_dow'),
    ]:
        out = pd.concat([out, frame], axis=1)

    # Historical frequency features: strictly prior events only.
    device_key = out['device_id'].fillna('Unknown')
    ip_key = out['ip_address'].astype('string').fillna('Unknown')
    out['device_prior_count'] = out.groupby(device_key, sort=False).cumcount()
    out['ip_prior_count'] = out.groupby(ip_key, sort=False).cumcount()
    out['log_device_prior_count'] = np.log1p(out['device_prior_count'])
    out['log_ip_prior_count'] = np.log1p(out['ip_prior_count'])
    out['device_seen_before'] = out['device_prior_count'].gt(0).astype('int8')
    out['ip_seen_before'] = out['ip_prior_count'].gt(0).astype('int8')

    return out.reset_index(drop=True)

features_df = build_event_time_features(fraud)

display(features_df.head())
print('Engineered shape:', features_df.shape)

## 7. Exploratory analysis

Fraud data are usually imbalanced. Accuracy can therefore look excellent even when the model misses most fraud. We first inspect prevalence, time drift, and the support behind segment-level rates.

In [ ]:
prevalence = features_df[TARGET].mean()
print(f'Fraud prevalence: {prevalence:.2%} ({features_df[TARGET].sum():,} / {len(features_df):,})')

ax = features_df[TARGET].value_counts().sort_index().plot(kind='bar', figsize=(7, 4))
ax.set(title='Target counts', xlabel='Class (0 = legitimate, 1 = fraud)', ylabel='Transactions')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
monthly = (
    features_df.set_index('purchase_time')[TARGET]
    .resample('MS')
    .agg(['count', 'sum', 'mean'])
    .rename(columns={'sum': 'fraud_n', 'mean': 'fraud_rate'})
)
display(monthly)

ax = monthly['fraud_rate'].plot(marker='o', figsize=(11, 4))
ax.axhline(prevalence, linestyle='--', label='Overall prevalence')
ax.set(title='Fraud rate over purchase time', xlabel='Purchase month', ylabel='Fraud rate')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def wilson_interval(successes: pd.Series, totals: pd.Series, z: float = 1.96):
    totals = totals.astype(float)
    p = successes / totals
    denom = 1 + z**2 / totals
    center = (p + z**2 / (2 * totals)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * totals)) / totals) / denom
    return center - margin, center + margin


def segment_fraud_table(df: pd.DataFrame, column: str, min_n: int = 100) -> pd.DataFrame:
    table = (
        df.groupby(column, dropna=False)[TARGET]
        .agg(transactions='size', fraud_n='sum', fraud_rate='mean')
        .reset_index()
    )
    table = table[table['transactions'] >= min_n].copy()
    table['ci_low'], table['ci_high'] = wilson_interval(
        table['fraud_n'], table['transactions']
    )
    return table.sort_values(['fraud_rate', 'transactions'], ascending=[False, False])

for col in ['source', 'browser', 'country', 'sex']:
    print(f'\n{col.upper()}')
    display(segment_fraud_table(features_df, col, min_n=100).head(15))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(data=features_df, x=TARGET, y='log_time_since_signup_hours', ax=axes[0], showfliers=False)
axes[0].set_title('Signup-to-purchase delay')

sns.boxplot(data=features_df, x=TARGET, y='log_device_prior_count', ax=axes[1], showfliers=False)
axes[1].set_title('Prior device activity')

sns.boxplot(data=features_df, x=TARGET, y='log_ip_prior_count', ax=axes[2], showfliers=False)
axes[2].set_title('Prior IP activity')

plt.tight_layout()
plt.show()

## 8. Modeling contract and leakage guardrails

### Prediction timestamp

The score is produced at the first purchase. Allowed features must exist by that moment.

### Default exclusions

- `class`: target.
- `user_id`: arbitrary identifier.
- `device_id`: raw high-cardinality identifier.
- `ip_address`: raw identifier; only mapped country and prior activity are used.
- `signup_time`, `purchase_time`: raw timestamps; only derived cyclic and elapsed-time features are used.
- `sex`: excluded from training by default and retained for subgroup auditing.

### Chronological partitions

1. **Train:** fit and compare algorithms.
2. **Calibration:** convert model scores into better probabilities.
3. **Threshold:** choose the business action threshold.
4. **Test:** untouched future period for final reporting.

This avoids training on future behavior and prevents threshold decisions from contaminating the test result.

In [ ]:
def chronological_split(
    df: pd.DataFrame,
    train_q: float = 0.60,
    calibration_q: float = 0.75,
    threshold_q: float = 0.85,
) -> dict[str, pd.DataFrame]:
    if not (0 < train_q < calibration_q < threshold_q < 1):
        raise ValueError('Quantiles must satisfy 0 < train < calibration < threshold < 1.')

    cutoffs = df['purchase_time'].quantile([train_q, calibration_q, threshold_q])
    t_train, t_cal, t_threshold = cutoffs.tolist()

    parts = {
        'train': df[df['purchase_time'] <= t_train].copy(),
        'calibration': df[(df['purchase_time'] > t_train) & (df['purchase_time'] <= t_cal)].copy(),
        'threshold': df[(df['purchase_time'] > t_cal) & (df['purchase_time'] <= t_threshold)].copy(),
        'test': df[df['purchase_time'] > t_threshold].copy(),
    }

    for name, part in parts.items():
        if part.empty or part[TARGET].nunique() < 2:
            raise ValueError(f'{name} split is empty or contains only one class.')
    return parts

parts = chronological_split(features_df)

split_summary = pd.DataFrame([
    {
        'split': name,
        'rows': len(part),
        'start': part['purchase_time'].min(),
        'end': part['purchase_time'].max(),
        'fraud_rate': part[TARGET].mean(),
        'fraud_n': int(part[TARGET].sum()),
    }
    for name, part in parts.items()
])
display(split_summary)

In [ ]:
NUMERIC_FEATURES = [
    'purchase_value', 'purchase_value_log', 'age',
    'time_since_signup_hours', 'log_time_since_signup_hours',
    'purchase_is_weekend', 'signup_is_weekend',
    'instant_purchase_10s', 'instant_purchase_1h',
    'purchase_hour_sin', 'purchase_hour_cos',
    'purchase_dow_sin', 'purchase_dow_cos',
    'signup_hour_sin', 'signup_hour_cos',
    'signup_dow_sin', 'signup_dow_cos',
    'device_prior_count', 'ip_prior_count',
    'log_device_prior_count', 'log_ip_prior_count',
    'device_seen_before', 'ip_seen_before',
]

CATEGORICAL_FEATURES = ['source', 'browser', 'country']
if USE_SENSITIVE_FEATURES:
    CATEGORICAL_FEATURES.append('sex')

MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

for forbidden in [TARGET, 'user_id', 'device_id', 'ip_address', 'signup_time', 'purchase_time']:
    assert forbidden not in MODEL_FEATURES

X = {name: part[MODEL_FEATURES].copy() for name, part in parts.items()}
y = {name: part[TARGET].copy() for name, part in parts.items()}

print(f'{len(MODEL_FEATURES)} model features')
print('Numeric:', NUMERIC_FEATURES)
print('Categorical:', CATEGORICAL_FEATURES)

## 9. XGBoost-native preprocessing and baseline models

### Native categorical representation

XGBoost can split categorical variables directly when:

- the inputs are pandas `category` columns;
- `enable_categorical=True`;
- the tree method is `hist` or `approx`.

This notebook fits a category schema only on each training window. Rare training categories are grouped into `__OTHER__`, missing categories use `__MISSING__`, and categories not seen in the training schema are mapped to `__OTHER__` during later scoring.

This avoids two problems:

1. a large one-hot matrix for country/browser/source combinations;
2. inconsistent integer category codes between training and prediction.

A dummy model and balanced logistic regression remain as sanity-check baselines. XGBoost is the intended final model in this notebook, but it should still demonstrate meaningful improvement over the baselines.

In [ ]:
MISSING_CATEGORY = '__MISSING__'
OTHER_CATEGORY = '__OTHER__'
MIN_CATEGORY_FREQUENCY = 10


def fit_category_schema(
    frame: pd.DataFrame,
    categorical_features: list[str],
    min_frequency: int = MIN_CATEGORY_FREQUENCY,
) -> dict[str, list[str]]:
    """Learn allowed category labels from one training window only."""
    schema: dict[str, list[str]] = {}
    for feature in categorical_features:
        values = frame[feature].astype('string').fillna(MISSING_CATEGORY)
        counts = values.value_counts(dropna=False)
        frequent = sorted(str(value) for value in counts[counts >= min_frequency].index)
        categories = list(dict.fromkeys(frequent + [MISSING_CATEGORY, OTHER_CATEGORY]))
        schema[feature] = categories
    return schema


def apply_category_schema(
    frame: pd.DataFrame,
    model_features: list[str],
    numeric_features: list[str],
    categorical_features: list[str],
    schema: dict[str, list[str]],
) -> pd.DataFrame:
    """Create a stable XGBoost DataFrame without using future category information."""
    out = frame[model_features].copy()

    for feature in numeric_features:
        out[feature] = pd.to_numeric(out[feature], errors='coerce').astype('float32')

    for feature in categorical_features:
        categories = schema[feature]
        known = set(categories)
        values = out[feature].astype('string').fillna(MISSING_CATEGORY)
        values = values.where(values.isin(known), OTHER_CATEGORY)
        out[feature] = pd.Categorical(values, categories=categories)

    return out


def class_balance_ratio(y_values: pd.Series | np.ndarray) -> float:
    y_array = np.asarray(y_values, dtype=int)
    positives = int(y_array.sum())
    negatives = int(len(y_array) - positives)
    if positives == 0:
        raise ValueError('A training window has no fraud examples.')
    return negatives / positives


def make_xgb_classifier(
    scale_pos_weight: float,
    n_estimators: int = 2_000,
    early_stopping_rounds: int | None = 75,
) -> xgb.XGBClassifier:
    """Create a regularized native-categorical XGBoost classifier."""
    return xgb.XGBClassifier(
        objective='binary:logistic',
        tree_method='hist',
        device=XGB_DEVICE,
        enable_categorical=True,
        max_cat_to_onehot=8,
        n_estimators=n_estimators,
        learning_rate=0.04,
        grow_policy='lossguide',
        max_depth=0,
        max_leaves=31,
        min_child_weight=10.0,
        gamma=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.10,
        reg_lambda=5.0,
        scale_pos_weight=scale_pos_weight,
        max_delta_step=1.0,
        max_bin=256,
        eval_metric=['logloss', 'aucpr'],  # The last metric controls early stopping.
        early_stopping_rounds=early_stopping_rounds,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )


linear_preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), NUMERIC_FEATURES),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(
            handle_unknown='infrequent_if_exist',
            min_frequency=MIN_CATEGORY_FREQUENCY,
            sparse_output=True,
        )),
    ]), CATEGORICAL_FEATURES),
])

baseline_models = {
    'Dummy prevalence': Pipeline([
        ('preprocess', linear_preprocessor),
        ('model', DummyClassifier(strategy='prior')),
    ]),
    'Balanced logistic regression': Pipeline([
        ('preprocess', linear_preprocessor),
        ('model', LogisticRegression(
            class_weight='balanced',
            C=0.5,
            max_iter=2_000,
            solver='lbfgs',
            random_state=RANDOM_STATE,
        )),
    ]),
}

print('Baseline models:', list(baseline_models))
print('XGBoost device:', XGB_DEVICE)

## 10. Rolling chronological validation with early stopping

`TimeSeriesSplit` creates expanding training windows and later validation windows.

For each XGBoost fold, the notebook:

1. learns the category schema from that fold's training rows;
2. maps the later validation rows using that training schema;
3. computes `scale_pos_weight = negatives / positives` from the fold's training labels;
4. trains up to 2,000 boosting rounds;
5. stops when validation `aucpr` has not improved for 75 rounds;
6. records the best boosting round and validation metrics.

The final model does **not** use the calibration or threshold periods for early stopping. Instead, the median best round from these historical folds becomes the final `n_estimators` value.

In [ ]:
def score_probability_metrics(
    y_true: pd.Series | np.ndarray,
    probability: np.ndarray,
) -> dict[str, float]:
    y_array = np.asarray(y_true, dtype=int)
    probability = np.clip(np.asarray(probability, dtype=float), 1e-7, 1 - 1e-7)
    return {
        'average_precision': average_precision_score(y_array, probability),
        'roc_auc': roc_auc_score(y_array, probability),
        'log_loss': log_loss(y_array, probability, labels=[0, 1]),
        'brier_loss': brier_score_loss(y_array, probability),
    }


tscv = TimeSeriesSplit(n_splits=3)
fold_rows: list[dict[str, float | int | str]] = []
xgb_best_rounds: list[int] = []
last_xgb_evals_result: dict[str, dict[str, list[float]]] | None = None

for fold, (train_idx, valid_idx) in enumerate(tscv.split(X['train']), start=1):
    X_fold_train = X['train'].iloc[train_idx]
    y_fold_train = y['train'].iloc[train_idx]
    X_fold_valid = X['train'].iloc[valid_idx]
    y_fold_valid = y['train'].iloc[valid_idx]

    # Baselines use complete scikit-learn pipelines, fitted only on the current past window.
    for model_name, estimator in baseline_models.items():
        fitted = clone(estimator)
        fitted.fit(X_fold_train, y_fold_train)
        probability = fitted.predict_proba(X_fold_valid)[:, 1]
        fold_rows.append({
            'fold': fold,
            'model': model_name,
            'train_rows': len(train_idx),
            'valid_rows': len(valid_idx),
            'best_rounds': np.nan,
            **score_probability_metrics(y_fold_valid, probability),
        })

    # XGBoost category vocabulary and class weighting are learned from this fold only.
    fold_schema = fit_category_schema(X_fold_train, CATEGORICAL_FEATURES)
    X_fold_train_xgb = apply_category_schema(
        X_fold_train, MODEL_FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES, fold_schema
    )
    X_fold_valid_xgb = apply_category_schema(
        X_fold_valid, MODEL_FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES, fold_schema
    )

    fold_pos_weight = class_balance_ratio(y_fold_train)
    fold_model = make_xgb_classifier(
        scale_pos_weight=fold_pos_weight,
        n_estimators=2_000,
        early_stopping_rounds=75,
    )
    fold_model.fit(
        X_fold_train_xgb,
        y_fold_train,
        eval_set=[
            (X_fold_train_xgb, y_fold_train),
            (X_fold_valid_xgb, y_fold_valid),
        ],
        verbose=False,
    )

    best_rounds = int(fold_model.best_iteration) + 1
    xgb_best_rounds.append(best_rounds)
    last_xgb_evals_result = fold_model.evals_result()
    xgb_probability = fold_model.predict_proba(X_fold_valid_xgb)[:, 1]

    fold_rows.append({
        'fold': fold,
        'model': 'Native categorical XGBoost',
        'train_rows': len(train_idx),
        'valid_rows': len(valid_idx),
        'best_rounds': best_rounds,
        'scale_pos_weight': fold_pos_weight,
        **score_probability_metrics(y_fold_valid, xgb_probability),
    })

fold_results = pd.DataFrame(fold_rows)
display(fold_results)

cv_summary = (
    fold_results.groupby('model', as_index=False)
    .agg(
        average_precision_mean=('average_precision', 'mean'),
        average_precision_std=('average_precision', 'std'),
        roc_auc_mean=('roc_auc', 'mean'),
        roc_auc_std=('roc_auc', 'std'),
        log_loss_mean=('log_loss', 'mean'),
        log_loss_std=('log_loss', 'std'),
        brier_loss_mean=('brier_loss', 'mean'),
        brier_loss_std=('brier_loss', 'std'),
    )
    .sort_values('average_precision_mean', ascending=False)
)
display(cv_summary)

FINAL_N_ESTIMATORS = int(np.clip(np.median(xgb_best_rounds), 25, 2_000))
SELECTED_MODEL_NAME = 'Native categorical XGBoost'

print('XGBoost best rounds by fold:', xgb_best_rounds)
print('Final number of boosting rounds:', FINAL_N_ESTIMATORS)

xgb_ap = cv_summary.loc[
    cv_summary['model'].eq('Native categorical XGBoost'), 'average_precision_mean'
].iloc[0]
best_baseline_ap = cv_summary.loc[
    ~cv_summary['model'].eq('Native categorical XGBoost'), 'average_precision_mean'
].max()
if xgb_ap <= best_baseline_ap:
    print(
        'Warning: XGBoost did not outperform the best baseline in rolling validation. '
        'Keep the comparison visible and investigate before deployment.'
    )

In [ ]:
if last_xgb_evals_result is not None:
    train_aucpr = last_xgb_evals_result['validation_0']['aucpr']
    valid_aucpr = last_xgb_evals_result['validation_1']['aucpr']
    train_logloss = last_xgb_evals_result['validation_0']['logloss']
    valid_logloss = last_xgb_evals_result['validation_1']['logloss']

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(train_aucpr, label='Train AUCPR')
    axes[0].plot(valid_aucpr, label='Validation AUCPR')
    axes[0].axvline(xgb_best_rounds[-1] - 1, linestyle='--', label='Best iteration')
    axes[0].set(title='Last rolling fold — AUCPR', xlabel='Boosting round', ylabel='AUCPR')
    axes[0].legend()

    axes[1].plot(train_logloss, label='Train log loss')
    axes[1].plot(valid_logloss, label='Validation log loss')
    axes[1].axvline(xgb_best_rounds[-1] - 1, linestyle='--', label='Best iteration')
    axes[1].set(title='Last rolling fold — log loss', xlabel='Boosting round', ylabel='Log loss')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## 11. Fit the final XGBoost model and calibrate its probabilities

The final category schema and class weight are learned from the full training partition.

The XGBoost model is then fitted for the historical median best number of boosting rounds. It does not use calibration, threshold, or test rows during model fitting.

### Sigmoid calibration

XGBoost may rank risk well while its numeric probabilities are too high or too low. A separate logistic sigmoid is fitted on the later calibration partition:

```text
raw XGBoost probability
→ log-odds transformation
→ one-variable logistic calibration model
→ calibrated fraud probability
```

The calibration partition is disjoint from model training. Calibration should mainly improve Brier and log loss; ranking metrics normally remain similar.

In [ ]:
# Learn stable categories from training only and apply them to all later periods.
CATEGORY_SCHEMA = fit_category_schema(X['train'], CATEGORICAL_FEATURES)
X_XGB = {
    name: apply_category_schema(
        frame, MODEL_FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES, CATEGORY_SCHEMA
    )
    for name, frame in X.items()
}

TRAIN_SCALE_POS_WEIGHT = class_balance_ratio(y['train'])
base_model = make_xgb_classifier(
    scale_pos_weight=TRAIN_SCALE_POS_WEIGHT,
    n_estimators=FINAL_N_ESTIMATORS,
    early_stopping_rounds=None,
)
base_model.fit(X_XGB['train'], y['train'], verbose=False)


def probability_to_logit(probability: np.ndarray, epsilon: float = 1e-6) -> np.ndarray:
    p = np.clip(np.asarray(probability, dtype=float), epsilon, 1 - epsilon)
    return np.log(p / (1 - p)).reshape(-1, 1)


raw_calibration_probability = base_model.predict_proba(X_XGB['calibration'])[:, 1]
calibrator = LogisticRegression(
    C=1_000_000.0,
    solver='lbfgs',
    max_iter=1_000,
    random_state=RANDOM_STATE,
)
calibrator.fit(probability_to_logit(raw_calibration_probability), y['calibration'])


def calibrate_raw_probability(raw_probability: np.ndarray) -> np.ndarray:
    return calibrator.predict_proba(probability_to_logit(raw_probability))[:, 1]


class CalibratedXGBoostView(ClassifierMixin, BaseEstimator):
    """In-memory scikit-compatible view for scoring and permutation importance."""

    def __init__(
        self,
        model,
        category_schema,
        calibrator,
        model_features,
        numeric_features,
        categorical_features,
    ):
        self.model = model
        self.category_schema = category_schema
        self.calibrator = calibrator
        self.model_features = model_features
        self.numeric_features = numeric_features
        self.categorical_features = categorical_features
        self.classes_ = np.array([0, 1])
        self.is_fitted_ = True

    def fit(self, frame=None, y=None):
        # The underlying XGBoost model and sigmoid calibrator are already fitted.
        return self

    def predict_proba(self, frame: pd.DataFrame) -> np.ndarray:
        transformed = apply_category_schema(
            frame,
            self.model_features,
            self.numeric_features,
            self.categorical_features,
            self.category_schema,
        )
        raw = self.model.predict_proba(transformed)[:, 1]
        calibrated = self.calibrator.predict_proba(probability_to_logit(raw))[:, 1]
        return np.column_stack([1 - calibrated, calibrated])

    def predict(self, frame: pd.DataFrame) -> np.ndarray:
        return (self.predict_proba(frame)[:, 1] >= 0.5).astype(int)


calibrated_model = CalibratedXGBoostView(
    model=base_model,
    category_schema=CATEGORY_SCHEMA,
    calibrator=calibrator,
    model_features=MODEL_FEATURES,
    numeric_features=NUMERIC_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
)

calibrated_calibration_probability = calibrated_model.predict_proba(X['calibration'])[:, 1]

calibration_comparison = pd.DataFrame([
    {
        'version': 'raw XGBoost',
        **score_probability_metrics(y['calibration'], raw_calibration_probability),
    },
    {
        'version': 'sigmoid calibrated',
        **score_probability_metrics(y['calibration'], calibrated_calibration_probability),
    },
])
display(calibration_comparison)

print('Training scale_pos_weight:', round(TRAIN_SCALE_POS_WEIGHT, 4))
print('Final boosting rounds:', FINAL_N_ESTIMATORS)
print('Calibrator coefficient:', float(calibrator.coef_[0, 0]))
print('Calibrator intercept:', float(calibrator.intercept_[0]))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
CalibrationDisplay.from_predictions(
    y['calibration'], raw_calibration_probability,
    n_bins=10, strategy='quantile', name='Raw XGBoost', ax=ax,
)
CalibrationDisplay.from_predictions(
    y['calibration'], calibrated_calibration_probability,
    n_bins=10, strategy='quantile', name='Sigmoid calibrated', ax=ax,
)
ax.set_title('Calibration-period reliability')
plt.tight_layout()
plt.show()

## 12. Tune a business threshold on a separate period

A classification threshold is an operating policy, not an inherent property of the model.

The example cost assumptions below are deliberately editable:

- A false negative represents missed fraud and can include chargeback, merchandise, processing, and investigation losses.
- A false positive represents review friction, customer support, conversion loss, or unnecessary authentication.

Replace these values with finance/operations estimates before making a real decision.

In [ ]:
FALSE_NEGATIVE_COST = 500.0
FALSE_POSITIVE_COST = 5.0
REVIEW_CAPACITY_RATE = 0.03


def threshold_metrics(
    y_true: pd.Series | np.ndarray,
    probability: np.ndarray,
    thresholds: Iterable[float],
    false_negative_cost: float,
    false_positive_cost: float,
) -> pd.DataFrame:
    y_array = np.asarray(y_true, dtype=int)
    rows = []
    for threshold in thresholds:
        pred = (probability >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_array, pred, labels=[0, 1]).ravel()
        total_cost = fp * false_positive_cost + fn * false_negative_cost
        rows.append({
            'threshold': threshold,
            'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
            'precision': precision_score(y_array, pred, zero_division=0),
            'recall': recall_score(y_array, pred, zero_division=0),
            'f1': f1_score(y_array, pred, zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(y_array, pred),
            'review_rate': pred.mean(),
            'total_cost': total_cost,
            'cost_per_1000': total_cost / len(y_array) * 1_000,
        })
    return pd.DataFrame(rows)

threshold_probability = calibrated_model.predict_proba(X['threshold'])[:, 1]
threshold_grid = np.unique(np.r_[np.linspace(0.001, 0.999, 500), 0.5])
threshold_table = threshold_metrics(
    y['threshold'], threshold_probability, threshold_grid,
    FALSE_NEGATIVE_COST, FALSE_POSITIVE_COST,
)

COST_THRESHOLD = float(
    threshold_table.loc[threshold_table['total_cost'].idxmin(), 'threshold']
)
CAPACITY_THRESHOLD = float(np.quantile(threshold_probability, 1 - REVIEW_CAPACITY_RATE))

policy_summary = threshold_metrics(
    y['threshold'],
    threshold_probability,
    [0.5, COST_THRESHOLD, CAPACITY_THRESHOLD],
    FALSE_NEGATIVE_COST,
    FALSE_POSITIVE_COST,
)
policy_summary.insert(0, 'policy', ['Default 0.5', 'Minimum expected cost', 'Review capacity'])
display(policy_summary)

print(f'Cost-optimal threshold: {COST_THRESHOLD:.4f}')
print(f'{REVIEW_CAPACITY_RATE:.1%} capacity threshold: {CAPACITY_THRESHOLD:.4f}')

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.plot(threshold_table['threshold'], threshold_table['precision'], label='Precision')
ax1.plot(threshold_table['threshold'], threshold_table['recall'], label='Recall')
ax1.plot(threshold_table['threshold'], threshold_table['review_rate'], label='Review rate')
ax1.axvline(COST_THRESHOLD, linestyle='--', label='Cost threshold')
ax1.axvline(CAPACITY_THRESHOLD, linestyle=':', label='Capacity threshold')
ax1.set(xlabel='Threshold', ylabel='Rate', title='Threshold trade-offs on policy-tuning period')
ax1.legend(loc='upper right')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(threshold_table['threshold'], threshold_table['cost_per_1000'])
ax.axvline(COST_THRESHOLD, linestyle='--', label='Minimum expected cost')
ax.set(xlabel='Threshold', ylabel='Assumed cost per 1,000 transactions', title='Cost curve')
ax.legend()
plt.tight_layout()
plt.show()

## 13. Final untouched future-period evaluation

The test period has not been used for model selection, calibration, or threshold choice. Both threshold-independent and threshold-dependent metrics are reported.

In [ ]:
def evaluate_binary_model(
    y_true: pd.Series | np.ndarray,
    probability: np.ndarray,
    threshold: float,
    false_negative_cost: float = FALSE_NEGATIVE_COST,
    false_positive_cost: float = FALSE_POSITIVE_COST,
) -> dict[str, float]:
    y_array = np.asarray(y_true, dtype=int)
    pred = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_array, pred, labels=[0, 1]).ravel()
    return {
        'threshold': threshold,
        'prevalence': y_array.mean(),
        'roc_auc': roc_auc_score(y_array, probability),
        'average_precision': average_precision_score(y_array, probability),
        'brier_loss': brier_score_loss(y_array, probability),
        'log_loss': log_loss(y_array, probability, labels=[0, 1]),
        'precision': precision_score(y_array, pred, zero_division=0),
        'recall': recall_score(y_array, pred, zero_division=0),
        'f1': f1_score(y_array, pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_array, pred),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'review_rate': pred.mean(),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'assumed_total_cost': fp * false_positive_cost + fn * false_negative_cost,
        'assumed_cost_per_1000': (
            (fp * false_positive_cost + fn * false_negative_cost) / len(y_array) * 1_000
        ),
    }

test_probability = calibrated_model.predict_proba(X['test'])[:, 1]
test_pred = (test_probability >= COST_THRESHOLD).astype(int)

test_metrics = pd.DataFrame([
    evaluate_binary_model(y['test'], test_probability, 0.5),
    evaluate_binary_model(y['test'], test_probability, COST_THRESHOLD),
    evaluate_binary_model(y['test'], test_probability, CAPACITY_THRESHOLD),
], index=['Default 0.5', 'Cost policy', 'Capacity policy'])

display(test_metrics.T)
print('\nClassification report — cost policy')
print(classification_report(y['test'], test_pred, digits=4, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y['test'], test_pred, labels=[0, 1], display_labels=['Legitimate', 'Fraud'],
    values_format=',d', ax=axes[0], colorbar=False,
)
axes[0].set_title(f'Confusion matrix — threshold {COST_THRESHOLD:.3f}')

precision, recall, _ = precision_recall_curve(y['test'], test_probability)
axes[1].plot(recall, precision, label=f'AP = {average_precision_score(y["test"], test_probability):.3f}')
axes[1].axhline(y['test'].mean(), linestyle='--', label='Prevalence baseline')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision–recall curve')
axes[1].legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y['test'], test_probability, ax=ax, name=SELECTED_MODEL_NAME)
ax.plot([0, 1], [0, 1], linestyle='--')
ax.set_title('ROC curve — future test period')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
CalibrationDisplay.from_predictions(
    y['test'], test_probability, n_bins=10, strategy='quantile', ax=ax,
    name='Calibrated model',
)
ax.set_title('Probability calibration — future test period')
plt.tight_layout()
plt.show()

## 14. Lift and review-capacity analysis

Fraud teams often investigate only a small fraction of transactions. Lift asks whether the highest-risk transactions contain substantially more fraud than a random sample of the same size.

In [ ]:
def lift_table(y_true: pd.Series | np.ndarray, probability: np.ndarray, bins: int = 10) -> pd.DataFrame:
    frame = pd.DataFrame({'y': np.asarray(y_true), 'probability': probability})
    frame = frame.sort_values('probability', ascending=False).reset_index(drop=True)
    frame['risk_group'] = pd.qcut(
        frame.index + 1,
        q=bins,
        labels=[f'{i + 1}' for i in range(bins)],
    )
    grouped = frame.groupby('risk_group', observed=True).agg(
        transactions=('y', 'size'),
        fraud_n=('y', 'sum'),
        fraud_rate=('y', 'mean'),
        mean_score=('probability', 'mean'),
        min_score=('probability', 'min'),
    ).reset_index()
    grouped['population_pct'] = grouped['transactions'] / grouped['transactions'].sum()
    grouped['fraud_capture_pct'] = grouped['fraud_n'] / grouped['fraud_n'].sum()
    grouped['cumulative_population_pct'] = grouped['population_pct'].cumsum()
    grouped['cumulative_fraud_capture_pct'] = grouped['fraud_capture_pct'].cumsum()
    grouped['lift_vs_average'] = grouped['fraud_rate'] / frame['y'].mean()
    return grouped

lift = lift_table(y['test'], test_probability, bins=10)
display(lift)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(lift['risk_group'].astype(str), lift['fraud_rate'])
axes[0].axhline(y['test'].mean(), linestyle='--', label='Overall prevalence')
axes[0].set(title='Fraud rate by risk decile', xlabel='Risk decile (1 = highest)', ylabel='Fraud rate')
axes[0].legend()

axes[1].plot(lift['cumulative_population_pct'], lift['cumulative_fraud_capture_pct'], marker='o')
axes[1].plot([0, 1], [0, 1], linestyle='--', label='Random selection')
axes[1].set(title='Cumulative gains', xlabel='Cumulative transactions reviewed', ylabel='Cumulative fraud captured')
axes[1].legend()
plt.tight_layout()
plt.show()

## 15. Permutation importance on calibrated future performance

Permutation importance measures how much future-test average precision decreases when one original business feature is shuffled.

This method evaluates the complete scoring view:

```text
category mapping → XGBoost → sigmoid calibration
```

It does not establish causality. Correlated raw/log or indicator features can share importance, and repeated test-driven feature changes would invalidate the test as a final holdout.

In [ ]:
importance_result = permutation_importance(
    calibrated_model,
    X['test'],
    y['test'],
    scoring='average_precision',
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance = pd.DataFrame({
    'feature': MODEL_FEATURES,
    'importance_mean': importance_result.importances_mean,
    'importance_std': importance_result.importances_std,
}).sort_values('importance_mean', ascending=False)

display(importance.head(20))

plot_imp = importance.head(20).sort_values('importance_mean')
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(plot_imp['feature'], plot_imp['importance_mean'], xerr=plot_imp['importance_std'])
ax.set(title='Permutation importance — average precision decrease', xlabel='Importance')
plt.tight_layout()
plt.show()

## 16. XGBoost-native interpretation

Two XGBoost-specific summaries are added:

### Gain importance

Gain measures the average training-loss improvement produced by splits using a feature. It is model-internal and can favor features that offer many possible split points.

### SHAP contribution summary

XGBoost can calculate additive feature contributions to the **raw model margin**. The notebook reports mean absolute contribution magnitude on a sample of future test rows.

The SHAP values explain the uncalibrated XGBoost margin. The later sigmoid calibration changes the probability scale, so these values should not be interpreted as direct percentage-point changes in calibrated fraud probability.

In [ ]:
gain_scores = base_model.get_booster().get_score(importance_type='gain')
gain_importance = (
    pd.DataFrame({
        'feature': MODEL_FEATURES,
        'gain': [gain_scores.get(feature, 0.0) for feature in MODEL_FEATURES],
    })
    .sort_values('gain', ascending=False)
)
display(gain_importance.head(20))

plot_gain = gain_importance.head(20).sort_values('gain')
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(plot_gain['feature'], plot_gain['gain'])
ax.set(title='XGBoost gain importance', xlabel='Average gain')
plt.tight_layout()
plt.show()

# Native SHAP contributions on a bounded sample for notebook runtime.
shap_sample_n = min(2_000, len(X_XGB['test']))
X_shap = X_XGB['test'].sample(shap_sample_n, random_state=RANDOM_STATE)
dmatrix_shap = xgb.DMatrix(X_shap, enable_categorical=True)
shap_contributions = base_model.get_booster().predict(dmatrix_shap, pred_contribs=True)

# The final column is the expected-value bias term.
mean_abs_shap = np.abs(shap_contributions[:, :-1]).mean(axis=0)
shap_importance = (
    pd.DataFrame({'feature': MODEL_FEATURES, 'mean_abs_raw_margin_shap': mean_abs_shap})
    .sort_values('mean_abs_raw_margin_shap', ascending=False)
)
display(shap_importance.head(20))

plot_shap = shap_importance.head(20).sort_values('mean_abs_raw_margin_shap')
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(plot_shap['feature'], plot_shap['mean_abs_raw_margin_shap'])
ax.set(title='Mean absolute native SHAP contribution', xlabel='Absolute contribution to raw margin')
plt.tight_layout()
plt.show()

## 16. Error analysis

Inspecting false negatives and false positives helps identify missing features, label problems, threshold issues, or process gaps. Raw identifiers are shown only for audit/debugging and must be access-controlled in a real system.

In [ ]:
audit_columns = [
    'user_id', 'purchase_time', 'purchase_value', 'country', 'source', 'browser', 'sex',
    'time_since_signup_hours', 'device_prior_count', 'ip_prior_count', TARGET,
]

audit = parts['test'][audit_columns].copy()
audit['fraud_probability'] = test_probability
audit['predicted_fraud'] = test_pred
audit['error_type'] = np.select(
    [
        (audit[TARGET] == 1) & (audit['predicted_fraud'] == 0),
        (audit[TARGET] == 0) & (audit['predicted_fraud'] == 1),
    ],
    ['false_negative', 'false_positive'],
    default='correct',
)

print('Highest-risk false positives')
display(
    audit[audit['error_type'] == 'false_positive']
    .sort_values('fraud_probability', ascending=False)
    .head(20)
)

print('Highest-score false negatives')
display(
    audit[audit['error_type'] == 'false_negative']
    .sort_values('fraud_probability', ascending=False)
    .head(20)
)

## 17. Subgroup audit

`sex` is not used by the default model, but it is retained to audit error rates. Country, source, and browser are also checked. Small groups are filtered because their rates are unstable.

A disparity is a signal for investigation—not automatic evidence of unlawful discrimination. Review label quality, sample size, customer exposure, operational policy, and local legal requirements.

In [ ]:
def subgroup_metrics(
    frame: pd.DataFrame,
    probability: np.ndarray,
    group: str,
    threshold: float,
    min_n: int = 100,
) -> pd.DataFrame:
    work = frame[[group, TARGET]].copy()
    work['probability'] = probability
    work['pred'] = (probability >= threshold).astype(int)

    rows = []
    for value, part in work.groupby(group, dropna=False):
        if len(part) < min_n or part[TARGET].nunique() < 2:
            continue
        tn, fp, fn, tp = confusion_matrix(part[TARGET], part['pred'], labels=[0, 1]).ravel()
        rows.append({
            group: value,
            'n': len(part),
            'prevalence': part[TARGET].mean(),
            'review_rate': part['pred'].mean(),
            'precision': precision_score(part[TARGET], part['pred'], zero_division=0),
            'recall': recall_score(part[TARGET], part['pred'], zero_division=0),
            'false_positive_rate': fp / (fp + tn) if fp + tn else np.nan,
            'average_precision': average_precision_score(part[TARGET], part['probability']),
        })
    return pd.DataFrame(rows).sort_values('n', ascending=False)

for group in ['sex', 'source', 'browser', 'country']:
    print(f'\n{group.upper()} AUDIT')
    display(subgroup_metrics(parts['test'], test_probability, group, COST_THRESHOLD, min_n=100).head(20))

## 18. Feature drift between training and future test periods

Population Stability Index (PSI) is included as a simple monitoring screen. It is a heuristic, not a statistical proof:

- below 0.10: usually stable;
- 0.10–0.25: watch;
- above 0.25: investigate.

Monitor score distributions, missingness, category changes, calibration, review rate, and delayed fraud outcomes in production.

In [ ]:
def _safe_distribution(values: pd.Series, categories: pd.Index, epsilon: float = 1e-6) -> np.ndarray:
    dist = values.value_counts(normalize=True, dropna=False).reindex(categories, fill_value=0).to_numpy()
    return np.clip(dist, epsilon, None)


def psi_categorical(reference: pd.Series, current: pd.Series) -> float:
    ref = reference.astype('string').fillna('Missing')
    cur = current.astype('string').fillna('Missing')
    categories = pd.Index(sorted(set(ref.unique()) | set(cur.unique())))
    p = _safe_distribution(ref, categories)
    q = _safe_distribution(cur, categories)
    return float(np.sum((q - p) * np.log(q / p)))


def psi_numeric(reference: pd.Series, current: pd.Series, bins: int = 10) -> float:
    ref = pd.to_numeric(reference, errors='coerce')
    cur = pd.to_numeric(current, errors='coerce')
    edges = np.unique(ref.quantile(np.linspace(0, 1, bins + 1)).to_numpy())
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    ref_bin = pd.cut(ref, edges, include_lowest=True).astype('string').fillna('Missing')
    cur_bin = pd.cut(cur, edges, include_lowest=True).astype('string').fillna('Missing')
    categories = pd.Index(sorted(set(ref_bin.unique()) | set(cur_bin.unique())))
    p = _safe_distribution(ref_bin, categories)
    q = _safe_distribution(cur_bin, categories)
    return float(np.sum((q - p) * np.log(q / p)))

psi_rows = []
for feature in MODEL_FEATURES:
    if feature in CATEGORICAL_FEATURES:
        value = psi_categorical(parts['train'][feature], parts['test'][feature])
    else:
        value = psi_numeric(parts['train'][feature], parts['test'][feature])
    psi_rows.append({'feature': feature, 'psi': value})

psi_table = pd.DataFrame(psi_rows).sort_values('psi', ascending=False)
psi_table['status'] = pd.cut(
    psi_table['psi'],
    bins=[-np.inf, 0.10, 0.25, np.inf],
    labels=['stable', 'watch', 'investigate'],
)
display(psi_table)

## 20. Save the native XGBoost model and decision-policy artifact

XGBoost's native categorical model is saved in JSON format. JSON/UBJSON model IO preserves native categorical metadata more reliably than treating the booster as a generic Python pickle.

A separate `joblib` artifact stores:

- category schema;
- sigmoid calibrator;
- decision and capacity thresholds;
- feature contract;
- cost assumptions;
- training/test windows;
- package versions;
- the relative JSON model filename.

The production system must reproduce the event-time country and historical device/IP features before scoring.

In [ ]:
XGB_MODEL_PATH = Path('fraud_xgboost_native_categorical.json')
POLICY_ARTIFACT_PATH = Path('fraud_xgboost_policy_artifact.joblib')
SCORED_TEST_PATH = Path('fraud_xgboost_scored_test.csv')

base_model.save_model(XGB_MODEL_PATH)

artifact = {
    'model_json_filename': XGB_MODEL_PATH.name,
    'selected_model_name': SELECTED_MODEL_NAME,
    'category_schema': CATEGORY_SCHEMA,
    'calibrator': calibrator,
    'decision_threshold': COST_THRESHOLD,
    'capacity_threshold': CAPACITY_THRESHOLD,
    'model_features': MODEL_FEATURES,
    'numeric_features': NUMERIC_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'uses_sensitive_features': USE_SENSITIVE_FEATURES,
    'xgboost_configuration': {
        'device_used_for_training': XGB_DEVICE,
        'n_estimators': FINAL_N_ESTIMATORS,
        'scale_pos_weight': TRAIN_SCALE_POS_WEIGHT,
        'tree_method': 'hist',
        'enable_categorical': True,
    },
    'cost_assumptions': {
        'false_negative_cost': FALSE_NEGATIVE_COST,
        'false_positive_cost': FALSE_POSITIVE_COST,
        'review_capacity_rate': REVIEW_CAPACITY_RATE,
    },
    'training_window': {
        'start': str(parts['train']['purchase_time'].min()),
        'end': str(parts['train']['purchase_time'].max()),
    },
    'test_window': {
        'start': str(parts['test']['purchase_time'].min()),
        'end': str(parts['test']['purchase_time'].max()),
    },
    'versions': {
        'python': platform.python_version(),
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'xgboost': xgb.__version__,
    },
}

joblib.dump(artifact, POLICY_ARTIFACT_PATH)
audit.to_csv(SCORED_TEST_PATH, index=False)

print(f'Saved XGBoost JSON model: {XGB_MODEL_PATH.resolve()}')
print(f'Saved calibration/policy artifact: {POLICY_ARTIFACT_PATH.resolve()}')
print(f'Saved scored test set: {SCORED_TEST_PATH.resolve()}')

In [ ]:
def score_engineered_transactions_xgboost(
    engineered: pd.DataFrame,
    policy_artifact_path: str | Path = POLICY_ARTIFACT_PATH,
) -> pd.DataFrame:
    """Score rows that already contain the event-time engineered features."""
    policy_path = Path(policy_artifact_path)
    saved = joblib.load(policy_path)

    missing = set(saved['model_features']).difference(engineered.columns)
    if missing:
        raise ValueError(f'Missing engineered features: {sorted(missing)}')

    model_path = policy_path.parent / saved['model_json_filename']
    if not model_path.exists():
        raise FileNotFoundError(f'XGBoost JSON model not found: {model_path}')

    loaded_model = xgb.XGBClassifier()
    loaded_model.load_model(model_path)

    transformed = apply_category_schema(
        engineered,
        saved['model_features'],
        saved['numeric_features'],
        saved['categorical_features'],
        saved['category_schema'],
    )
    raw_probability = loaded_model.predict_proba(transformed)[:, 1]
    calibrated_probability = saved['calibrator'].predict_proba(
        probability_to_logit(raw_probability)
    )[:, 1]

    result = pd.DataFrame(index=engineered.index)
    result['raw_xgboost_probability'] = raw_probability
    result['fraud_probability'] = calibrated_probability
    result['send_to_review'] = calibrated_probability >= saved['decision_threshold']
    result['capacity_policy_review'] = calibrated_probability >= saved['capacity_threshold']
    return result

score_engineered_transactions_xgboost(parts['test'].head())

## 21. Conclusions and recommended next steps

### Why XGBoost can work well here

- It models nonlinear thresholds in signup delay, transaction amount, and entity-history counts.
- It learns interactions among timing, device/IP history, browser, source, and country.
- Native categorical splits avoid a potentially large one-hot matrix.
- Histogram training is efficient for large tabular datasets.
- Fold-specific `scale_pos_weight` helps the minority fraud class influence training.
- Early stopping controls boosting complexity using later data inside each historical training fold.

### Why XGBoost is not automatically the correct production choice

- It must outperform simple baselines on repeated future windows.
- Its probabilities must remain calibrated after prevalence changes.
- Native categorical inputs and unknown-category behavior must be tested in serving.
- A complex tree model can be harder to explain and govern than logistic regression.
- Fraudsters adapt after the model changes the review process.

### Production checklist

1. Confirm the exact scoring timestamp and availability of every feature.
2. Implement prior device/IP counters in an event-time feature store.
3. Version the IP-to-country database and category schema.
4. Recalculate historical features without future events during every backtest.
5. Validate false-positive and false-negative costs with fraud operations, finance, and support.
6. Select a policy using economics, review capacity, and customer friction.
7. Use reason codes and a human-review/appeal path for adverse actions.
8. Monitor input drift, unseen categories, score drift, calibration, workload, and delayed labels.
9. Backtest on several future windows and shadow-score before enforcement.
10. Save and deploy the JSON model and policy artifact as one versioned release.

### Suggested XGBoost experiments

- Tune `max_leaves`, `min_child_weight`, `learning_rate`, `subsample`, and regularization using only rolling training-period validation.
- Compare native categories with one-hot encoding under the same time splits.
- Add 1-hour, 24-hour, and 7-day point-in-time velocity features.
- Use row-level expected fraud loss based on purchase value instead of one constant false-negative cost.
- Evaluate multi-action policies: approve, authenticate, review, and hold.
- Compare XGBoost with CatBoost or LightGBM only after preserving the same leakage-safe evaluation design.

The final decision should be based on future-period economics and operational stability—not on whether XGBoost is a more fashionable algorithm.

## Design references

- XGBoost: native categorical data and automatic recoding  
  https://xgboost.readthedocs.io/en/stable/tutorials/categorical.html
- XGBoost: scikit-learn estimator interface and early stopping  
  https://xgboost.readthedocs.io/en/stable/python/sklearn_estimator.html
- XGBoost: parameters, including `scale_pos_weight`  
  https://xgboost.readthedocs.io/en/stable/parameter.html
- XGBoost: parameter tuning for imbalanced data  
  https://xgboost.readthedocs.io/en/stable/tutorials/param_tuning.html
- XGBoost: stable JSON/UBJSON model serialization  
  https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html
- Scikit-learn: probability calibration  
  https://scikit-learn.org/stable/modules/calibration.html
- Scikit-learn: precision-recall analysis  
  https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html
- Scikit-learn: time-ordered cross-validation  
  https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html
- Scikit-learn: cost-sensitive threshold learning  
  https://scikit-learn.org/stable/auto_examples/model_selection/plot_cost_sensitive_learning.html
- Imbalanced-learn: avoiding leakage during resampling  
  https://imbalanced-learn.org/stable/common_pitfalls.html